In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

C:\Users\amrit\AppData\Local\Temp\ipykernel_38548\2611793992.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [3]:
from langchain.tools import tool

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

c:\Users\amrit\Documents\Projects\agent_experiments\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model="mistral-small-latest",
    tools=[sql_query]
)

In [7]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")

response = agent.invoke(
    {"messages": [question]}
)

In [8]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?", additional_kwargs={}, response_metadata={}, id='cf18e816-b34e-4177-835e-79c1233933da'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'fZYq8NaYU', 'type': 'function', 'function': {'name': 'sql_query', 'arguments': '{"query": "SELECT artist_name, COUNT(*) as play_count FROM tracks WHERE artist_name LIKE \'S%\' GROUP BY artist_name ORDER BY play_count DESC LIMIT 1;"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 92, 'total_tokens': 133, 'completion_tokens': 41, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-small-latest', 'model': 'mistral-small-latest', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019ef295-1e7f-7cf0-9870-94555eeb7298-0', tool_calls=[{'name': 'sql_query', 'args': {'query': "SELECT artist_name, COUNT(*) as play_count FROM tracks WHERE artist_name LIKE 'S%' GROUP BY artist_name ORDER BY 

In [9]:
print(response["messages"][-3].tool_calls[0]['args']['query'])

SELECT a.Name, COUNT(t.TrackId) as track_count FROM Artist a JOIN Album al ON a.ArtistId = al.ArtistId JOIN Track t ON al.AlbumId = t.AlbumId WHERE a.Name LIKE 'S%' GROUP BY a.Name ORDER BY track_count DESC LIMIT 1;
